# NTS0611 Processing

Generate

P(Trip Purpose | Age, Sex)

## Step 1 – Load the NTS Mobility Dataset


In [1]:
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
nts0611a = pd.read_excel('../dataset(mobility)/nts0611 Average number of trips and distance travelled by sex, age and purpose England.ods', 
                         sheet_name='NTS0611a_trips', skiprows=5)

In [3]:
display(nts0611a.head())

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],Other [note 1],All purposes,Unweighted sample size: individuals,Unweighted sample size: trips (thousands)
0,2002,All people,0 to 16,10.965709,0.601430,252.435245,41.212326,84.532346,158.556695,72.385670,153.334289,26.625333,51.058601,32.746723,11.683989,22.478070,23.582358,0.075075,942.273859,3084,52.582
1,2002,All people,17 to 20,220.875052,9.504783,121.107835,6.799904,158.791795,37.123523,70.541268,176.830659,92.210300,46.763126,16.793644,6.681502,10.534957,20.952200,7.288901,1002.799449,547,10.115
2,2002,All people,21 to 29,268.906081,36.833463,33.816748,37.922372,199.521342,76.936998,91.762702,138.665827,71.354870,47.723286,17.696085,9.364377,14.812031,15.517718,0.389899,1061.223796,1262,24.934
3,2002,All people,30 to 39,254.569086,69.237394,6.745660,115.046571,235.796917,131.253956,120.152378,120.541820,52.938785,43.707804,15.600966,9.101992,26.115206,31.236774,0.165421,1232.210729,2083,47.089
4,2002,All people,40 to 49,289.872426,70.669378,2.979659,80.692897,256.576587,139.028234,133.807504,105.243938,54.731065,45.300523,15.269404,12.874706,22.186667,49.604000,0.000000,1278.836987,2011,47.071


We can check what are the trip purposes in this dataset

In [4]:
nts0611a.columns

Index(['Year', 'Sex [note 2]', 'Age', 'Commuting', 'Business', 'Education',
       'Escort education', 'Shopping', 'Other escort', 'Personal business',
       'Visiting friends at private home', 'Visiting friends elsewhere',
       'Entertainment or public activity', 'Sport: participate',
       'Holiday: base', 'Day trip', 'Just walk [note 1]', 'Other [note 1]',
       'All purposes', 'Unweighted sample size: individuals',
       'Unweighted sample size: trips (thousands)'],
      dtype='object')

## Step 2 – Data Preprocessing

Before constructing the trip-purpose probability matrix, irrelevant variables are removed from the dataset.

Only the demographic attributes (**Year**, **Age**, and **Sex**) and trip-purpose variables are retained for subsequent analysis. Summary statistics (e.g., sample size), total trip counts, and the **Other** category are excluded because they do not contribute to the proposed probabilistic framework. The remaining trip-purpose categories are preserved to maintain consistency with the original NTS ontology before the subsequent probability transformation.

### Select Relevant Variables

In [5]:
# Working copy
nts0611a_processed = nts0611a.copy()

# The study year is 2024 only, so we can add a column for the year and set it to 2024 for all rows.
nts0611a_processed = nts0611a_processed[nts0611a_processed["Year"] == 2024].copy()

nts0611a_selected_columns = [
    "Year",
    "Sex [note 2]",
    "Age",
    "Commuting",
    "Business",
    "Education",
    "Escort education",
    "Shopping",
    "Other escort",
    "Personal business",
    "Visiting friends at private home",
    "Visiting friends elsewhere",
    "Entertainment or public activity",
    "Sport: participate",
    "Holiday: base",
    "Day trip",
    "Just walk [note 1]"]

nts0611a_processed = nts0611a_processed[nts0611a_selected_columns]


### Define Trip Purpose Variables

To ensure that subsequent calculations are applied only to behavioural variables, the trip-purpose columns are explicitly specified. This list is reused throughout the framework for probability normalisation, temporal integration, and the subsequent purpose-to-POI mapping.

In [6]:
trip_purpose_columns = [
    "Commuting",
    "Business",
    "Education",
    "Escort education",
    "Shopping",
    "Other escort",
    "Personal business",
    "Visiting friends at private home",
    "Visiting friends elsewhere",
    "Entertainment or public activity",
    "Sport: participate",
    "Holiday: base",
    "Day trip",
    "Just walk [note 1]"
]

## Step 3 – Construct Trip Purpose Probabilities

The NTS0611 dataset reports the **average number of trips per person per year** for each trip purpose rather than probabilities.

To incorporate these observations into the proposed probabilistic framework, the annual trip frequencies are normalised within each demographic group. This converts the observed trip frequencies into a probability distribution over trip purposes while preserving the relative travel behaviour reported by the National Travel Survey.

The resulting probability matrix represents:

\[
P(Trip Purpose|Age,Sex)
\]

where the probabilities across all trip purposes sum to one for each demographic group.

In [7]:
nts0611a_processed["Total Trips"] = (
    nts0611a_processed[trip_purpose_columns].sum(axis=1))

In [8]:
nts0611a_processed[["Age", "Sex [note 2]", "Total Trips"]].head()

,Age,Sex [note 2],Total Trips
594,0 to 16,All people,804.003353
595,17 to 20,All people,728.974963
596,21 to 29,All people,849.416147
597,30 to 39,All people,1022.621548
598,40 to 49,All people,1047.105936


In [9]:
trip_purpose_probability = nts0611a_processed.copy()

trip_purpose_probability[trip_purpose_columns] = (trip_purpose_probability[trip_purpose_columns].div(trip_purpose_probability["Total Trips"], axis=0))

### Verify

In [10]:
if np.all(np.isclose(
    trip_purpose_probability[trip_purpose_columns].sum(axis=1),
    1.0
)):
    print("All trip purpose probabilities sum to 1.")
else:
    print("Some rows do not sum to 1.")

All trip purpose probabilities sum to 1.


In [11]:
trip_purpose_probability.head()

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],Total Trips
594,2024,All people,0 to 16,0.003350,0.000610,0.330359,0.046021,0.027737,0.141211,0.071645,0.099516,0.040360,0.072901,0.051326,0.013606,0.045030,0.056329,804.003353
595,2024,All people,17 to 20,0.163109,0.008059,0.187075,0.006645,0.160894,0.027464,0.074342,0.107256,0.074952,0.084809,0.019193,0.014184,0.028536,0.043483,728.974963
596,2024,All people,21 to 29,0.241196,0.032758,0.032333,0.024868,0.183033,0.039831,0.071758,0.087714,0.061615,0.073876,0.014004,0.020335,0.035252,0.081428,849.416147
597,2024,All people,30 to 39,0.184012,0.040306,0.002677,0.125048,0.176102,0.101364,0.071759,0.069205,0.046484,0.054593,0.011323,0.012867,0.034799,0.069460,1022.621548
598,2024,All people,40 to 49,0.166936,0.037345,0.001661,0.140342,0.180166,0.098854,0.076576,0.053395,0.043312,0.056311,0.012028,0.012163,0.037616,0.083295,1047.105936


In [12]:
trip_purpose_probability.describe()

,Year,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],Total Trips
count,27.0,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,2024.0,0.128520,0.025426,0.069329,0.053110,0.186725,0.074460,0.090620,0.085127,0.056859,0.066269,0.018494,0.014356,0.041315,0.089391,913.595258
std,0.0,0.079822,0.016997,0.111023,0.048574,0.075247,0.034817,0.026590,0.018928,0.014122,0.014427,0.013739,0.002469,0.008918,0.030673,117.296128
min,2024.0,0.003106,0.000525,0.000087,0.005370,0.023888,0.025216,0.067392,0.047336,0.037175,0.050995,0.006094,0.010744,0.020385,0.042978,683.129771
25%,2024.0,0.080555,0.007706,0.000615,0.014063,0.175636,0.044786,0.072299,0.072599,0.045564,0.057724,0.010643,0.012829,0.035868,0.068288,813.324580
50%,2024.0,0.139083,0.028163,0.002677,0.044123,0.182868,0.072517,0.077210,0.087714,0.053757,0.063735,0.014304,0.014130,0.041132,0.090420,921.212153
75%,2024.0,0.178584,0.037406,0.070729,0.064975,0.207668,0.097261,0.095609,0.097966,0.062828,0.072957,0.019768,0.014990,0.046118,0.120751,1010.695886
max,2024.0,0.265123,0.057559,0.333408,0.171169,0.324676,0.142315,0.159272,0.120002,0.091820,0.120107,0.065606,0.021265,0.062934,0.135993,1113.561566


## Step 4 - Age Harmonisation

The age groups adopted in the proposed demographic ontology are different from those used in the National Travel Survey (NTS). Therefore, the mobility probabilities reported by the NTS cannot be directly assigned to the ontology age groups.

To estimate mobility probabilities for the proposed age groups, single-year age population data from the Office for National Statistics (ONS) Census were used to calculate the population proportion within each overlapping age interval. These population proportions were then used as weights to derive population-weighted mobility probabilities for the harmonised age groups.

The ONS Census was selected because it provides single-year age distributions, allowing the overlap between different age classifications to be quantified. In contrast, the NTS only reports aggregated age groups and therefore cannot provide the weighting information required for harmonisation.

This harmonisation preserves the behavioural information contained in the NTS while ensuring consistency with the demographic representation used throughout the framework.

A simple arithmetic average was not adopted because the target age groups overlap multiple NTS age intervals with different population sizes. Population-weighted averaging provides a more representative estimate by accounting for the underlying demographic distribution.

In [13]:
age_dataset = pd.read_excel("../dataset(household)/ONS_Age.xlsx")

new_age_dataset = age_dataset[["Lower tier local authorities", "Age (86 categories) Code", "Observation"]].copy()

In [14]:
study_area = "Manchester"

age_single = new_age_dataset[new_age_dataset["Lower tier local authorities"] == study_area].copy()
age_single.head()

,Lower tier local authorities,Age (86 categories) Code,Observation
20812,Manchester,0,6581
20813,Manchester,1,7057
20814,Manchester,2,6819
20815,Manchester,3,6782
20816,Manchester,4,7139


### Step 4.1: Age mapping dictionary

In [15]:
age_mapping = {
    "0-4": [{"nts": "0 to 16", "start": 0, "end": 4}],

    "5-11": [{"nts": "0 to 16", "start": 5, "end": 11}],

    "12-17": [
        {"nts": "0 to 16", "start": 12, "end": 16},
        {"nts": "17 to 20", "start": 17, "end": 17},
    ],

    "18-22": [
        {"nts": "17 to 20", "start": 18, "end": 20},
        {"nts": "21 to 29", "start": 21, "end": 22},
    ],

    "23-34": [
        {"nts": "21 to 29", "start": 23, "end": 29},
        {"nts": "30 to 39", "start": 30, "end": 34},
    ],

    "35-49": [
        {"nts": "30 to 39", "start": 35, "end": 39},
        {"nts": "40 to 49", "start": 40, "end": 49},
    ],

    "50-64": [
        {"nts": "50 to 59", "start": 50, "end": 59},
        {"nts": "60 to 69", "start": 60, "end": 64},
    ],

    "65-74": [
        {"nts": "60 to 69", "start": 65, "end": 69},
        {"nts": "70 and over", "start": 70, "end": 74},
    ],
    
    "75+": [{"nts": "70 and over", "start": 75, "end": None}],
}

### Step  4.2: Create a generator to calculate population

In [16]:
def get_population_count(start_age, end_age=None):
    #Open-ended age groups (e.g. 75+) are handled by aggregating all population counts above the specified lower bound.

    if end_age is None:

        return age_single[
            age_single["Age (86 categories) Code"] >= start_age
        ]["Observation"].sum()

    return age_single[
        age_single["Age (86 categories) Code"].between(start_age, end_age)
    ]["Observation"].sum()

### Step 4.3: Verify

In [17]:
print(get_population_count(23, 29))
print(get_population_count(30, 34))

75439
48001


In [18]:
group = age_mapping["23-34"]

populations = []

for part in group:
    pop = get_population_count(part["start"], part["end"])
    populations.append(pop)

total = sum(populations)

weights = [p / total for p in populations]

print(weights)

[np.float64(0.6111390149060272), np.float64(0.38886098509397277)]


In [19]:
def get_weights_for_framework_age(framework_age):
    
    #Calculate population-based interpolation weights for one framework age group.

    age_overlap = age_mapping[framework_age]

    populations = []

    for overlap in age_overlap:
        population = get_population_count(
            overlap["start"],
            overlap["end"]
        )
        populations.append(population)

    total_population = sum(populations)

    overlap_weights = [
        population / total_population
        for population in populations
    ]

    return overlap_weights

In [20]:
display(get_weights_for_framework_age("23-34"))

[np.float64(0.6111390149060272), np.float64(0.38886098509397277)]

In [21]:
def harmonise_one_age_group(trip_purpose_probability, framework_age, sex, trip_purpose_columns):
    
    """Harmonise one framework age group using population-weighted interpolation."""

    age_overlap = age_mapping[framework_age]

    overlap_weights = get_weights_for_framework_age(
        framework_age
    )

    harmonised_trip_purpose_probability = pd.Series(
        0.0,
        index=trip_purpose_columns
    )

    for overlap, weight in zip(age_overlap, overlap_weights):

        nts_row = trip_purpose_probability[
            (trip_purpose_probability["Sex [note 2]"] == sex)
            &
            (trip_purpose_probability["Age"] == overlap["nts"])
        ].iloc[0]

        harmonised_trip_purpose_probability += (
            weight * nts_row[trip_purpose_columns]
        )

    harmonised_row = {
        "Year": 2024,
        "Sex [note 2]": sex,
        "Age": framework_age
    }

    harmonised_row.update(
        harmonised_trip_purpose_probability.to_dict()
    )

    harmonised_row["prob_sum"] = (
        harmonised_trip_purpose_probability.sum()
    )

    return harmonised_row

In [22]:
framework_age_groups = [
    "0-4",
    "5-11",
    "12-17",
    "18-22",
    "23-34",
    "35-49",
    "50-64",
    "65-74",
    "75+"
]

sex_groups = [
    "Females",
    "Males"
]

harmonised_rows = []

for sex in sex_groups:

    for framework_age in framework_age_groups:

        harmonised_rows.append(

            harmonise_one_age_group(
                trip_purpose_probability,
                framework_age,
                sex,
                trip_purpose_columns
            )

        )

harmonised_trip_purpose_probability = pd.DataFrame(
    harmonised_rows
)

harmonised_trip_purpose_probability = (
    harmonised_trip_purpose_probability[
        ["Year", "Sex [note 2]", "Age"]
        + trip_purpose_columns
        + ["prob_sum"]
    ]
)

display(harmonised_trip_purpose_probability)

,Year,Sex [note 2],Age,Commuting,Business,Education,Escort education,Shopping,Other escort,Personal business,Visiting friends at private home,Visiting friends elsewhere,Entertainment or public activity,Sport: participate,Holiday: base,Day trip,Just walk [note 1],prob_sum
0,2024,Females,0-4,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662,1.0
1,2024,Females,5-11,0.003616,0.000701,0.327535,0.044123,0.031882,0.142315,0.075573,0.106521,0.043625,0.078185,0.035980,0.015151,0.041132,0.053662,1.0
2,2024,Females,12-17,0.026672,0.001690,0.304515,0.038551,0.056038,0.125005,0.074419,0.108590,0.051023,0.074196,0.031747,0.015177,0.040354,0.052022,1.0
3,2024,Females,18-22,0.180319,0.015662,0.117592,0.019347,0.189261,0.036292,0.071061,0.109007,0.080139,0.057312,0.007461,0.017746,0.035592,0.063210,1.0
4,2024,Females,23-34,0.187671,0.030108,0.019513,0.088612,0.184447,0.069928,0.074421,0.086093,0.055916,0.059392,0.006704,0.017174,0.035667,0.084355,1.0
5,2024,Females,35-49,0.138959,0.030173,0.002271,0.170156,0.180556,0.100192,0.075522,0.064854,0.043113,0.053496,0.009181,0.011911,0.036452,0.083165,1.0
6,2024,Females,50-64,0.131147,0.032095,0.000690,0.039488,0.222626,0.066860,0.105210,0.088050,0.052823,0.062339,0.008032,0.013860,0.046023,0.130756,1.0
7,2024,Females,65-74,0.045730,0.013056,0.000397,0.021163,0.285039,0.047241,0.141740,0.109224,0.070948,0.075429,0.008485,0.013501,0.049347,0.118700,1.0
8,2024,Females,75+,0.011872,0.003727,0.000087,0.012801,0.324676,0.032908,0.159272,0.113154,0.082330,0.078310,0.007110,0.012737,0.047206,0.113809,1.0
9,2024,Males,0-4,0.003106,0.000525,0.333408,0.047855,0.023888,0.139715,0.068052,0.093075,0.037175,0.067786,0.065606,0.012175,0.048738,0.058896,1.0


### Rationale for Trip Purpose Mapping

The trip purposes defined in the National Travel Survey (NTS) are linked to the POI ontology adopted in this framework to establish a semantic relationship between observed travel behaviour and spatial destinations represented by the OS Points of Interest (POI) dataset.

At this stage, the mapping is not used to produce final POI probabilities. Instead, it defines the possible destination categories associated with each trip purpose. This distinction is important because some trip purposes can be directly mapped to a single POI category, while others require additional demographic or spatial information before a final POI category can be assigned.

Three mapping strategies were employed:

### 1. Direct mapping 
Direct mapping was applied where a trip purpose corresponded to a single POI category. For example, Shopping was mapped to Retail, Education and Escort education were mapped to Education, and Holiday: base was mapped to Accommodation.

### 2. Ambiguous mapping 
Ambiguous mapping was applied where a trip purpose may correspond to multiple POI categories. Personal business may correspond to Healthcare, Personal Services, or Public Services. Rather than equally distributing this probability during the Mobility Layer, the probability is retained at the trip-purpose level and resolved later using the Opportunity Layer.

### 3. Conditional mapping 
Conditional mapping was applied to employment-related trips. Commuting and Business are treated as employment-related trip purposes, but they are not directly fixed to a single POI category. During destination resolution, these trips are refined using the individual’s industry information, allowing employment-related trips to be associated with different workplace POI categories, such as healthcare, retail, education, or office-related locations.

### Transport
Transport infrastructure is represented separately within the Opportunity Layer because transport facilities function as mobility-enabling infrastructure rather than final trip-purpose destinations.

Several trip purposes were excluded from the mapping process. Visiting friends at private home was excluded because private residential locations are not represented within the adopted POI ontology. Other escort and Other were excluded due to insufficient semantic information to infer a reliable destination category. All purposes was excluded because it represents an aggregate summary of all trips rather than an individual trip purpose.

Accordingly, the output of the Mobility Layer remains a trip-purpose probability distribution:

P(Trip Purpose | Age, Sex)

POI categories are treated as semantic destination candidates and are resolved in later layers, rather than being fixed during the Mobility Layer.

### Personal business

Personal business is treated as an ambiguous trip purpose because it may correspond to multiple destination categories, including Healthcare, Personal Services, and Public Services.

Rather than equally distributing this probability during the Mobility Layer, the probability is retained at the trip-purpose level. The final allocation is deferred to the Opportunity Layer, where the relative availability of Healthcare, Personal Services, and Public Services POIs can be used to resolve the destination category.

This avoids introducing arbitrary allocation assumptions before spatial opportunity information is considered.

| **POI Ontology**         | **Mapped Trip Purpose(s)**                                                                            | **Mapping Strategy**    | **Justification**                                                                                                                                           |
| ------------------------ | ----------------------------------------------------------------------------------------------------- | ----------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Employment               | Commuting, Business                                                                                   | Conditional mapping     | Employment destinations are further refined using individual industry information during destination allocation.                                            |
| Education                | Education, Escort education                                                                           | Direct mapping          | Both purposes explicitly correspond to educational facilities.                                                                                              |
| Retail                   | Shopping                                                                                              | Direct mapping          | Shopping activities are directly associated with retail destinations.                                                                                       |
| Healthcare               | Personal business                                                                                     | Ambiguous Mapping       | Personal business includes medical consultations and healthcare services.                                                                                   |
| Personal Services        | Personal business                                                                                     | Ambiguous Mapping       | Personal business includes visits to personal service providers such as hairdressers and dry-cleaners.                                                      |
| Public Services          | Personal business                                                                                     | Ambiguous Mapping       | Personal business includes visits to banks, libraries, churches and other public service facilities.                                                        |
| Food and Beverage        | Visiting friends elsewhere, Entertainment or public activity, Sport: participate                      | Ambiguous Mapping       | These leisure activities may reasonably occur in food and beverage venues; equal probability is assumed due to the aggregated nature of the NTS categories. |
| Recreation               | Visiting friends elsewhere, Entertainment or public activity, Sport: participate, Day trip, Just walk | Ambiguous Mapping       | These trip purposes primarily represent leisure and recreational activities occurring outside the home.                                                     |
| Accommodation            | Holiday: base                                                                                         | Direct mapping          | Holiday trips involving overnight stays are directly associated with accommodation facilities.                                                              |
| Transport Infrastructure | —                                                                                                     | Separate representation | Transport facilities are represented independently within the Opportunity Layer rather than inferred from trip purposes.                                    |


### Step 6.1 - Established the POI columns

In [23]:
poi_columns = [
    "Employment",
    "Education",
    "Retail",
    "Healthcare",
    "Personal Services",
    "Public Services",
    "Food and Beverage",
    "Recreation",
    "Accommodation"
]

In [24]:
poi_probability = harmonised_trip_purpose_probability[
    ["Year", "Sex [note 2]", "Age"]].copy()

for poi in poi_columns:
    poi_probability[poi] = 0.0

In [25]:
display(poi_probability.head())

,Year,Sex [note 2],Age,Employment,Education,Retail,Healthcare,Personal Services,Public Services,Food and Beverage,Recreation,Accommodation
0,2024,Females,0-4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024,Females,5-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024,Females,12-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024,Females,18-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024,Females,23-34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
purpose_to_poi_candidates = {
    "Shopping": ["Retail"],
    "Education": ["Education"],
    "Escort education": ["Education"],
    "Holiday: base": ["Accommodation"],

    # Personal business may correspond to multiple destination types.
    # The final POI category is determined later by the Opportunity Layer.
    "Personal business": [
        "Healthcare",
        "Personal Services",
        "Public Services"
    ],

    # Employment-related trips cannot be assigned to a single POI category.
    # Individuals working in different industries travel to different workplace POIs.
    # For example, healthcare workers travel to healthcare facilities,
    # teachers travel to education facilities, and retail workers travel to retail locations.
    # Therefore, Commuting and Business trips are resolved later using the individual's industry.
    "Commuting": ["Industry-dependent"],
    "Business": ["Industry-dependent"],

    "Visiting friends elsewhere": [
        "Food and Beverage",
        "Recreation"
    ],
    "Entertainment or public activity": [
        "Food and Beverage",
        "Recreation"
    ],
    "Sport: participate": [
        "Food and Beverage",
        "Recreation"
    ],
    "Day trip": ["Recreation"],
    "Just walk [note 1]": ["Recreation"]
}

# The Mobility Layer determines WHY people travel (trip purpose),
# while the final workplace destination is determined by combining
# trip purpose with demographic information (Industry) in the
# destination resolution stage.

In [27]:
mapping_rows = []

for purpose, candidates in purpose_to_poi_candidates.items():
    for candidate in candidates:
        mapping_rows.append({
            "Trip Purpose": purpose,
            "Candidate POI Category": candidate
        })

purpose_to_poi_mapping_df = pd.DataFrame(mapping_rows)

display(purpose_to_poi_mapping_df)

,Trip Purpose,Candidate POI Category
0,Shopping,Retail
1,Education,Education
2,Escort education,Education
3,Holiday: base,Accommodation
4,Personal business,Healthcare
5,Personal business,Personal Services
6,Personal business,Public Services
7,Commuting,Industry-dependent
8,Business,Industry-dependent
9,Visiting friends elsewhere,Food and Beverage
